In [1]:

import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
from shapely import wkt
import torchvision.transforms.functional as TF
from torchvision.models import ResNet50_Weights

Build an Extract , Transform and Load Pipeline for the model

In [ ]:

# Initialize classes of interest

DAMAGE_CLASS_MAP = {
    "no-damage": 1, "minor-damage": 2, "major-damage": 3,
    "destroyed": 4, "un-classified": 0,
}

NUM_CLASSES = 5

_weights = ResNet50_Weights.DEFAULT.transforms()
NORM_MEAN, NORM_STD = _weights.mean, _weights.std


# File discovery

def discover_image_ids(root: str):
    """Map image_id -> its disaster folder, for every post-disaster label under root."""
    lookup = {}
    for label_path in Path(root).rglob("*_post_disaster.json"):
        image_id = label_path.stem.replace("_post_disaster", "")
        lookup[image_id] = label_path.parent.parent
    return lookup


# Loading raw files

def load_image(disaster_dir: Path, image_id: str, phase: str):
    return Image.open(disaster_dir / "images" / f"{image_id}_{phase}_disaster.png").convert("RGB")


def load_label(disaster_dir: Path, image_id: str, phase: str):
    with open(disaster_dir / "labels" / f"{image_id}_{phase}_disaster.json") as f:
        return json.load(f)


# Mask building

def build_damage_mask(label: dict, size: tuple):
    """Rasterize each building polygon (already in this chip's pixel coords) to its damage class."""
    mask = Image.new("L", size, 0)
    draw = ImageDraw.Draw(mask)
    for feature in label.get("features", {}).get("xy", []):
        subtype = feature.get("properties", {}).get("subtype", "un-classified")
        polygon = wkt.loads(feature["wkt"])
        draw.polygon(list(polygon.exterior.coords), fill=DAMAGE_CLASS_MAP.get(subtype, 0))
    return np.array(mask, dtype=np.int64)


#  Resize / augment

def resize_sample(pre, post, mask, size: int):
    """Nearest-neighbor for the mask, so resizing never blends two class labels."""
    pre = pre.resize((size, size), Image.BILINEAR)
    post = post.resize((size, size), Image.BILINEAR)
    mask = np.array(Image.fromarray(mask.astype(np.uint8)).resize((size, size), Image.NEAREST), dtype=np.int64)
    return pre, post, mask


def augment_sample(pre, post, mask):
    """Same random flip/rotation applied to all three, so the mask stays aligned."""
    if random.random() < 0.5:
        pre, post, mask = TF.hflip(pre), TF.hflip(post), TF.hflip(mask.unsqueeze(0)).squeeze(0)
    if random.random() < 0.5:
        pre, post, mask = TF.vflip(pre), TF.vflip(post), TF.vflip(mask.unsqueeze(0)).squeeze(0)
    angle = random.choice([0, 90, 180, 270])
    if angle:
        pre, post = TF.rotate(pre, angle), TF.rotate(post, angle)
        mask = TF.rotate(mask.unsqueeze(0), angle).squeeze(0)
    return pre, post, mask


# Sample assembly

def get_sample(disaster_dir: Path, image_id: str, image_size: int, augment: bool):
    """Load -> rasterize -> resize -> (augment) -> normalize one pre/post pair."""
    pre_img = load_image(disaster_dir, image_id, "pre")
    post_img = load_image(disaster_dir, image_id, "post")
    mask = build_damage_mask(load_label(disaster_dir, image_id, "post"), post_img.size)
    pre_img, post_img, mask = resize_sample(pre_img, post_img, mask, image_size)

    pre, post, mask = TF.to_tensor(pre_img), TF.to_tensor(post_img), torch.from_numpy(mask)
    if augment:
        pre, post, mask = augment_sample(pre, post, mask)
    pre, post = TF.normalize(pre, NORM_MEAN, NORM_STD), TF.normalize(post, NORM_MEAN, NORM_STD)

    return {"pre_image": pre, "post_image": post, "mask": mask, "image_id": image_id}


#  Dataset class

class XBDDataset(Dataset):
    """Thin wrapper — all logic lives in get_sample()."""

    def __init__(self, image_ids, disaster_lookup, image_size=512, augment=False):
        self.image_ids, self.disaster_lookup = image_ids, disaster_lookup
        self.image_size, self.augment = image_size, augment

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        return get_sample(self.disaster_lookup[image_id], image_id, self.image_size, self.augment)


# Train/val split

def split_ids_by_event(disaster_lookup: dict, val_fraction: float = 0.15, seed: int = 42):
    """Split within each disaster event, so no rare event lands entirely in one split."""
    rng = random.Random(seed)
    by_event = {}
    for image_id, disaster_dir in disaster_lookup.items():
        by_event.setdefault(disaster_dir.name, []).append(image_id)

    train_ids, val_ids = [], []
    for ids in by_event.values():
        ids = ids.copy()
        rng.shuffle(ids)
        n_val = max(1, int(len(ids) * val_fraction))
        val_ids += ids[:n_val]
        train_ids += ids[n_val:]
    return train_ids, val_ids


# Class weights

def compute_class_weights(dataset: Dataset, max_samples: int = 200):
    """Inverse-frequency weights for CrossEntropyLoss, addressing the no-damage class skew."""
    counts = torch.zeros(NUM_CLASSES, dtype=torch.float64)
    for i in range(min(max_samples, len(dataset))):
        mask = dataset[i]["mask"]
        for c in range(NUM_CLASSES):
            counts[c] += (mask == c).sum()
    counts = counts.clamp(min=1)
    return (counts.sum() / (NUM_CLASSES * counts)).float()


# DataLoaders

def build_dataloaders(root: str, batch_size: int = 8, image_size: int = 512,
                       val_fraction: float = 0.15, num_workers: int = 4, seed: int = 42):
    """Raw folder in, train/val DataLoaders out."""
    disaster_lookup = discover_image_ids(root)
    train_ids, val_ids = split_ids_by_event(disaster_lookup, val_fraction, seed)

    train_ds = XBDDataset(train_ids, disaster_lookup, image_size, augment=True)
    val_ds = XBDDataset(val_ids, disaster_lookup, image_size, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=num_workers)
    return train_loader, val_loader, train_ds, val_ds


# Entry point

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True)
    parser.add_argument("--batch-size", type=int, default=8)
    args = parser.parse_args()

    train_loader, val_loader, train_ds, val_ds = build_dataloaders(args.data, batch_size=args.batch_size)
    print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches")
    print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")
    print(f"Class weights: {compute_class_weights(train_ds, max_samples=100).tolist()}")